In [2]:
# Install dependencies
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install pandas numpy tables matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.0 MB/s eta 0:00:00


In [3]:
# Import libraries and set random seed
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, BatchNorm, GatedGraphConv
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
import zipfile

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Extract dataset
zip_path = 'METR-LA.zip'
data_dir = 'METR-LA'
if not os.path.exists(data_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
print(f"Data directory {data_dir} ready")

Data directory METR-LA ready


In [5]:
# Load METR-LA data for 99 days
def load_metr_la_data(data_dir='METR-LA', num_days=99):
    adj_mat = np.load(os.path.join(data_dir, 'adj_mat.npy'))  # Shape: [207, 207]
    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))  # Shape: [34272, 207, 2]
    print(f"Original node_values shape: {node_values.shape}")
    # Select first 99 days (28512 time steps)
    time_steps = num_days * 288  # 288 time steps per day
    node_values = node_values[:time_steps]  # Shape: [28512, 207, 2]
    if node_values.ndim == 3:
        node_values = node_values[:, :, 0]  # Shape: [28512, 207]
        node_values = node_values.T  # Shape: [207, 28512]
    elif node_values.ndim != 2:
        raise ValueError(f"Expected 2D or 3D node_values, got shape {node_values.shape}")
    print(f"Processed node_values shape: {node_values.shape}")
    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T  # Shape: [207, 28512]
    edge_index = np.where(adj_mat > 0)
    edge_index = torch.tensor(np.array(edge_index), dtype=torch.long)  # Keep on CPU
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)  # Keep on CPU
    # Move to device after all NumPy operations
    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)
    print(f"edge_index device: {edge_index.device}, shape: {edge_index.shape}")
    print(f"edge_weight device: {edge_weight.device}, shape: {edge_weight.shape}")
    return node_values, edge_index, edge_weight, scaler

# Run the function
node_values, edge_index, edge_weight, scaler = load_metr_la_data()

Original node_values shape: (34272, 207, 2)
Processed node_values shape: (207, 28512)
edge_index device: cuda:0, shape: torch.Size([2, 1722])
edge_weight device: cuda:0, shape: torch.Size([1722])


In [6]:
# Create data samples with 60-20-20 split
def create_data_samples(node_values, in_timestep=12, out_timestep=12):
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = torch.tensor(features, dtype=torch.float).permute(0, 2, 1)
    labels = torch.tensor(labels, dtype=torch.float).permute(0, 2, 1)
    return features, labels

# Run the function
features, labels = create_data_samples(node_values)

# Custom 60-20-20 split
total_samples = len(features)
train_end = int(0.6 * total_samples)
val_end = train_end + int(0.2 * total_samples)

train_features = features[:train_end]
train_labels = labels[:train_end]
val_features = features[train_end:val_end]
val_labels = labels[train_end:val_end]
test_features = features[val_end:]
test_labels = labels[val_end:]

print(f"Train samples: {len(train_features)}, Val samples: {len(val_features)}, Test samples: {len(test_features)}")

<ipython-input-6-4157497971>:7: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  features = torch.tensor(features, dtype=torch.float).permute(0, 2, 1)


Train samples: 17093, Val samples: 5697, Test samples: 5699


In [7]:
# Create data for the 100th day (time steps 28512 to 28799)
def create_100th_day_data(data_dir='METR-LA', in_timestep=12, out_timestep=12):
    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))
    node_values = node_values[28512:28512+288]  # 100th day
    node_values = node_values[:, :, 0].T  # Shape: [207, 288]
    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = torch.tensor(features, dtype=torch.float).permute(0, 2, 1)
    labels = torch.tensor(labels, dtype=torch.float).permute(0, 2, 1)
    return features, labels, scaler

day_100_features, day_100_labels, day_100_scaler = create_100th_day_data()

In [8]:
# Prepare DataLoader objects
transform = AddRandomWalkPE(walk_length=20, attr_name='pe')

# Training, validation, and test data
train_data = []
for i in range(len(train_features)):
    x = train_features[i].T  # Shape: [207, 12]
    y = train_labels[i].T    # Shape: [207, 12]
    data = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_weight)
    data = transform(data)
    train_data.append(data)

val_data = []
for i in range(len(val_features)):
    x = val_features[i].T
    y = val_labels[i].T
    data = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_weight)
    data = transform(data)
    val_data.append(data)

test_data = []
for i in range(len(test_features)):
    x = test_features[i].T
    y = test_labels[i].T
    data = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_weight)
    data = transform(data)
    test_data.append(data)

# 100th day data
day_100_data = []
for i in range(len(day_100_features)):
    x = day_100_features[i].T
    y = day_100_labels[i].T
    data = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_weight)
    data = transform(data)
    day_100_data.append(data)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
day_100_loader = DataLoader(day_100_data, batch_size=32, shuffle=False)

In [9]:
# Define GPSLayer
class GPSLayer(nn.Module):
    def __init__(self, hidden_channels, heads=4):
        super(GPSLayer, self).__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=0.1)
        self.norm = BatchNorm(hidden_channels)

    def forward(self, x, edge_index, edge_weight, batch):
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        x_trans = self.transformer(x_gcn, edge_index)
        x = self.norm(x_gcn + x_trans)
        x = torch.relu(x)
        return x

In [10]:
# Define GPSGraphTransformer
class GPSGraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, pe_dim=20, heads=4):
        super().__init__()
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim)
        self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.layers = nn.ModuleList([
            GPSLayer(hidden_channels, heads) for _ in range(num_layers)
        ])
        self.norm = BatchNorm(hidden_channels)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )

    def forward(self, x, edge_index, edge_weight, pe, batch):
        x = self.node_emb(x)
        pe = self.pe_lin(pe)
        x = torch.cat([x, pe], dim=-1)
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
        x = self.norm(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)

In [11]:
# Compute evaluation metrics
def compute_metrics(pred, target, scaler):
    batch_size, num_nodes, out_timestep = pred.shape
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0
    mape = torch.where(mask, torch.abs((pred_unscaled - target_unscaled) / (target_unscaled + 1e-2)), 0.0)
    mape = torch.mean(mape[mask]) * 100 if mask.sum() > 0 else 0.0
    return mae, rmse, mape

In [12]:
# Visualize single batch predictions
def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5, filename='prediction_100th_day.png'):
    os.makedirs(output_dir, exist_ok=True)
    pred = pred[0].cpu().numpy().T
    target = target[0].cpu().numpy().T
    pred = scaler.inverse_transform(pred).T
    target = scaler.inverse_transform(target).T
    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred.shape[0])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred[i], label='Predicted', marker='o')
        plt.plot(target[i], label='Actual', marker='x')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (100th Day)')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

In [13]:
# Train and evaluate model
model = GPSGraphTransformer(
    in_channels=12, hidden_channels=64, out_channels=12, num_layers=4, pe_dim=20, heads=4
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
criterion = nn.MSELoss()

for epoch in range(20):
    model.train()
    total_train_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
        batch_size = data.batch.max().item() + 1
        num_nodes = out.shape[1]
        target = data.y.view(batch_size, num_nodes, -1)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            loss = criterion(out, target)
            total_val_loss += loss.item()
    val_loss = total_val_loss / len(val_loader)
    print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# Evaluate on test set
model.eval()
total_mae, total_rmse, total_mape = 0, 0, 0
num_batches = 0
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
        batch_size = data.batch.max().item() + 1
        num_nodes = out.shape[1]
        target = data.y.view(batch_size, num_nodes, -1)
        mae, rmse, mape = compute_metrics(out, target, scaler)
        total_mae += mae
        total_rmse += rmse
        total_mape += mape
        num_batches += 1
        if num_batches == 1:
            visualize_predictions(out, target, scaler, output_dir='plots_test', filename='prediction_test.png')

print(f"Test MAE: {total_mae / num_batches:.4f} mph")
print(f"Test RMSE: {total_rmse / num_batches:.4f} mph")
print(f"Test MAPE: {total_mape / num_batches:.4f}%")

# Evaluate on 100th day
total_mae, total_rmse, total_mape = 0, 0, 0
num_batches = 0
with torch.no_grad():
    for data in day_100_loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
        batch_size = data.batch.max().item() + 1
        num_nodes = out.shape[1]
        target = data.y.view(batch_size, num_nodes, -1)
        mae, rmse, mape = compute_metrics(out, target, day_100_scaler)
        total_mae += mae
        total_rmse += rmse
        total_mape += mape
        num_batches += 1
        if num_batches == 1:
            visualize_predictions(out, target, day_100_scaler)

print(f"100th Day MAE: {total_mae / num_batches:.4f} mph")
print(f"100th Day RMSE: {total_rmse / num_batches:.4f} mph")
print(f"100th Day MAPE: {total_mape / num_batches:.4f}%")

Epoch 1, Train Loss: 0.0511, Val Loss: 0.0206
Epoch 2, Train Loss: 0.0247, Val Loss: 0.0201
Epoch 3, Train Loss: 0.0244, Val Loss: 0.0196
Epoch 4, Train Loss: 0.0240, Val Loss: 0.0194
Epoch 5, Train Loss: 0.0239, Val Loss: 0.0200
Epoch 6, Train Loss: 0.0236, Val Loss: 0.0192
Epoch 7, Train Loss: 0.0235, Val Loss: 0.0194
Epoch 8, Train Loss: 0.0233, Val Loss: 0.0232
Epoch 9, Train Loss: 0.0232, Val Loss: 0.0209
Epoch 10, Train Loss: 0.0230, Val Loss: 0.0198
Epoch 11, Train Loss: 0.0228, Val Loss: 0.0222
Epoch 12, Train Loss: 0.0226, Val Loss: 0.0182
Epoch 13, Train Loss: 0.0225, Val Loss: 0.0206
Epoch 14, Train Loss: 0.0227, Val Loss: 0.0187
Epoch 15, Train Loss: 0.0223, Val Loss: 0.0204
Epoch 16, Train Loss: 0.0222, Val Loss: 0.0193
Epoch 17, Train Loss: 0.0222, Val Loss: 0.0207
Epoch 18, Train Loss: 0.0219, Val Loss: 0.0189
Epoch 19, Train Loss: 0.0217, Val Loss: 0.0184
Epoch 20, Train Loss: 0.0215, Val Loss: 0.0185
Test MAE: 5.8173 mph
Test RMSE: 9.2399 mph
Test MAPE: 12.0611%
100th 

In [16]:
# Visualize full 100th day predictions
def visualize_full_100th_day(day_100_loader, model, scaler, output_dir='plots', num_sensors=5):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for data in day_100_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    # Concatenate all predictions and targets
    all_preds = np.concatenate(all_preds, axis=0)  # Shape: [265, 207, 12]
    all_targets = np.concatenate(all_targets, axis=0)  # Shape: [265, 207, 12]

    # Reconstruct full day (288 time steps)
    num_nodes = all_preds.shape[1]
    full_preds = np.zeros((num_nodes, 288))
    full_targets = np.zeros((num_nodes, 288))
    counts = np.zeros((num_nodes, 288))

    for i in range(all_preds.shape[0]):
        start_idx = i + 12  # Input window is 12 time steps
        end_idx = start_idx + 12
        full_preds[:, start_idx:end_idx] += all_preds[i]  # Shape: [207, 12]
        full_targets[:, start_idx:end_idx] += all_targets[i]  # Shape: [207, 12]
        counts[:, start_idx:end_idx] += 1

    # Average overlapping predictions, avoiding division by zero
    mask = counts > 0
    full_preds[mask] = full_preds[mask] / counts[mask]
    full_targets[mask] = full_targets[mask] / counts[mask]
    # Leave zeros where counts == 0 (time steps 0–11 and 277–287)

    # Inverse scale
    full_preds = scaler.inverse_transform(full_preds.T).T
    full_targets = scaler.inverse_transform(full_targets.T).T

    # Plot only covered time steps (12 to 276)
    covered_time_steps = slice(12, 277)  # Time steps 12 to 276
    time_steps = np.arange(12, 277) * 5 / 60  # Convert to hours
    os.makedirs(output_dir, exist_ok=True)
    plt.figure(figsize=(15, 10))
    for i in range(min(num_sensors, num_nodes)):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(time_steps, full_preds[i, covered_time_steps], label='Predicted', marker='o', markersize=3)
        plt.plot(time_steps, full_targets[i, covered_time_steps], label='Actual', marker='x', markersize=3)
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (100th Day, Hours 1–23)')
        plt.xlabel('Time (hours)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'full_100th_day_prediction.png'))
    plt.close()

# Run the visualizer
visualize_full_100th_day(day_100_loader, model, day_100_scaler)